In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Dell\\OneDrive\\Desktop\\mlops-project\\chest-ct-ecs\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Dell\\OneDrive\\Desktop\\mlops-project\\chest-ct-ecs'

In [5]:
!pip install dagshub -q
print("✅ DagsHub installed")

✅ DagsHub installed


In [9]:
import os
import sys
import mlflow
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd() / "src"))

# Set MLflow tracking to your DagsHub repository
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/Gajju9191/chest-ct-ecs.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "Gajju9191"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "089e1f4ec33ad67cc8541160fe89a199ce77186d"

print("✅ MLflow tracking configured!")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

✅ MLflow tracking configured!
Tracking URI: https://dagshub.com/Gajju9191/chest-ct-ecs.mlflow


In [10]:
# Test if MLflow can connect
try:
    with mlflow.start_run(run_name="MLflow Connection Test"):
        mlflow.log_param("test_param", "working")
        mlflow.log_metric("test_metric", 0.99)
        print("✅ MLflow is working and logging to DagsHub!")
except Exception as e:
    print(f"❌ Error: {e}")

✅ MLflow is working and logging to DagsHub!


In [11]:
import tensorflow as tf

In [12]:
model = tf.keras.models.load_model("artifacts/training/model.h5")
print("✅ Model loaded successfully!")
print(f"Model type: {type(model)}")



✅ Model loaded successfully!
Model type: <class 'keras.src.engine.functional.Functional'>


In [13]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int
    params_classes: int

In [14]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

print("✅ Imports successful!")
print(f"CONFIG_FILE_PATH: {CONFIG_FILE_PATH}")
print(f"PARAMS_FILE_PATH: {PARAMS_FILE_PATH}")
print(f"save_json function: {save_json}")

✅ Imports successful!
CONFIG_FILE_PATH: config\config.yaml
PARAMS_FILE_PATH: params.yaml
save_json function: <function save_json at 0x00000188AFA7F4C0>


In [15]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model=Path("artifacts/training/model.h5"),
            training_data=Path("artifacts/data_ingestion/Chest-CT-Scan-data"),
            mlflow_uri="https://dagshub.com/Gajju9191/chest-ct-ecs.mlflow",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE,
            params_classes=self.params.CLASSES
        )
        return eval_config

print("✅ ConfigurationManager defined for notebook experiment!")

✅ ConfigurationManager defined for notebook experiment!


In [16]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

In [17]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def _valid_generator(self):
        datagenerator_kwargs = dict(
            rescale=1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = self.model.evaluate(self.valid_generator)  # Fixed: changed model to self.model
        self.save_score()

    def save_score(self):
        scores = {"loss": float(self.score[0]), "accuracy": float(self.score[1])}
        save_json(path=Path("scores.json"), data=scores)

    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics({
                "loss": float(self.score[0]), 
                "accuracy": float(self.score[1])
            })

            # Model registry does not work with file store
            if tracking_url_type_store != "file":
                # Register the model
                mlflow.keras.log_model(
                    self.model, 
                    "model", 
                    registered_model_name="VGG16Model"
                )
            else:
                mlflow.keras.log_model(self.model, "model")

print("✅ Evaluation class defined!")

✅ Evaluation class defined!


In [18]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()
    print("\n✅ Evaluation completed and logged to MLflow successfully!")
    
except Exception as e:
    print(f"❌ Error: {e}")
    raise e

[2026-05-11 12:19:02,422: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-11 12:19:02,438: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-11 12:19:02,438: INFO: common: created directory at: artifacts]
Found 102 images belonging to 2 classes.

[2026-05-11 12:19:04,165: WARNING: module_wrapper: From c:\Users\Dell\anaconda3\envs\chest-ct-ecs\Lib\site-packages\keras\src\utils\tf_utils.py:492: The name tf.ragged.RaggedTensorValue is deprecated. Please use tf.compat.v1.ragged.RaggedTensorValue instead.
]
9/9 [==============================] - 19s 2s/step - loss: 0.3464 - accuracy: 0.8137
[2026-05-11 12:19:22,057: INFO: common: json file saved at: scores.json]


2026/05/11 12:19:24 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


INFO:tensorflow:Assets written to: C:\Users\Dell\AppData\Local\Temp\tmp9kypaxg7\model\data\model\assets
[2026-05-11 12:19:26,290: INFO: builder_impl: Assets written to: C:\Users\Dell\AppData\Local\Temp\tmp9kypaxg7\model\data\model\assets]


c:\Users\Dell\anaconda3\envs\chest-ct-ecs\Lib\site-packages\_distutils_hack\__init__.py:30: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(
Successfully registered model 'VGG16Model'.
2026/05/11 12:20:12 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: VGG16Model, version 1
Created version '1' of model 'VGG16Model'.



✅ Evaluation completed and logged to MLflow successfully!
